In [13]:
import multilayerGM as gm

n_nodes = 50
n_layers = 2
p = 1.0
theta = 0.2 
n_sets = 5
dt = gm.dependency_tensors.UniformMultiplex(n_nodes, n_layers, p)
null = gm.DirichletNull(layers=dt.shape[1:], theta=theta, n_sets=n_sets)

partition = gm.sample_partition(dependency_tensor=dt, null_distribution=null)
multinet = gm.multilayer_DCSBM_network(partition, mu=0.0, k_min=2, k_max=10, t_k=-2)

/Users/acw721/Desktop/research/linkstream/.venv/lib/python3.12/site-packages/multilayerGM/networks.py:93: RuntimeWarning: invalid value encountered in divide
  sigma = [degrees[g] / sum(degrees[g]) for g in partition_map]


In [14]:
print("Generated multilayer network with", multinet.number_of_edges(), "edges", multinet.number_of_nodes(), "nodes")

Generated multilayer network with 170 edges 100 nodes


In [15]:
import networkx as nx
import numpy as np
import pandas as pd

def save_graph_csv_poisson_int_ts(
    G: nx.Graph,
    out_csv: str,
    lam: float = 10.0,
    comm_attr: str = "mesoset",
    seed: int | None = 42,
    base_time: int = 0,
    sort_by_time: bool = True,
    make_strictly_increasing: bool = False,
    layer_index: int = 1,
    enforce_same_layer: bool = True,
):
    rng = np.random.default_rng(seed)

    edges = []
    if G.is_multigraph():
        for u, v, k in G.edges(keys=True):
            edges.append((u, v))
    else:
        for u, v in G.edges():
            edges.append((u, v))

    df_cols = ["source", "destination", "timestamp", "source_commu", "destination_commu"]
    m = len(edges)
    if m == 0:
        pd.DataFrame(columns=df_cols).to_csv(out_csv, index=False)
        return

    edges_by_layer: dict[int, list[tuple]] = {}
    for (u, v) in edges:
        lu = u[layer_index] if isinstance(u, tuple) and len(u) > layer_index else None
        lv = v[layer_index] if isinstance(v, tuple) and len(v) > layer_index else None

        if enforce_same_layer and (lu != lv):
            raise ValueError(f"Edge endpoints are in different layers: {u} vs {v}")

        layer = lu
        edges_by_layer.setdefault(layer, []).append((u, v))


    shuffled_edges = []
    for layer in sorted(edges_by_layer.keys(), key=lambda x: (x is None, x)):
        layer_edges = edges_by_layer[layer]
        rng.shuffle(layer_edges)
        shuffled_edges.extend(layer_edges)

    ts = rng.poisson(lam=lam, size=len(shuffled_edges)).astype(np.int64)

    if make_strictly_increasing:
        ts = np.cumsum(ts + 1).astype(np.int64)

    ts = (ts + np.int64(base_time)).astype(np.int64)

    rows = []
    for i, (u, v) in enumerate(shuffled_edges):
        rows.append({
            "source": u[0] if isinstance(u, tuple) else u,
            "destination": v[0] if isinstance(v, tuple) else v,
            "timestamp": int(ts[i]),
            "source_commu": G.nodes[u].get(comm_attr, None),
            "destination_commu": G.nodes[v].get(comm_attr, None),
        })

    df = pd.DataFrame(rows)
    df["timestamp"] = df["timestamp"].astype("int64")

    if sort_by_time:
        df = df.sort_values(["timestamp"], kind="stable").reset_index(drop=True)

    df.to_csv(out_csv, index=False)

save_graph_csv_poisson_int_ts(
    multinet,
    out_csv="syn_net_test.csv",
    lam=5.0,
    comm_attr="mesoset",
    seed=42,
    base_time=0,
    sort_by_time=True,
    make_strictly_increasing=True,
)

In [4]:
from GraphSynthesiser import GraphSynthesiser

synth = GraphSynthesiser()

n_nodes = 50
n_layers = 5
p_list = [0.8] # prob nodes keep commu label across layers
theta = 0.2
n_sets = 5
mu_list = [0.2] # mu = 0 -> no inter-community links

for p in p_list:
    for mu in mu_list:
        for i in range(1,5):
            synth.synthesize_to_csv(
                f"syn_data/syn_net_p{p}_mu{mu}_{i}.csv",
                n_nodes=n_nodes,
                n_layers=n_layers,
                p=p,
                theta=theta,
                n_sets=n_sets,
                mu=mu,
                k_min=2,
                k_max=10,
                t_k=-2,
                lam=5.0,
                seed=42,
                base_time=0,
                sort_by_time=True,
                make_strictly_increasing=True,
            )